# Práctica de detección de objetos — Fase de mejoras

### **Grupo:** N — **Integrantes:** Lucas Amado Oseira y María Lestón Geada
##### **Dataset:** Lettuce Pallets (`Francesco/lettuce-pallets`)
##### **Detectores:** Cascade R-CNN (Detectron2) y SSD300-VGG16 (Torchvision), ambos preentrenados en MS COCO

## 0. Cómo usar este cuaderno

Este cuaderno continúa el de baselines y reutiliza su estructura: las mismas rutas (`/kaggle/working/work`), la misma limpieza de datos, el mismo `evaluate_coco`, el mismo informe impreso por `print_metrics`, el mismo protocolo de tiempos (`time_inference`) y el mismo `results.csv`, al que se añade una fila por experimento y split. Se ejecuta de arriba abajo en Kaggle con la GPU y el acceso a Internet activados.

Los baselines no se reentrenan si ya existen. Para reutilizar los del cuaderno anterior hay que añadir su salida como *Input* de este cuaderno (*Add Input → Notebook Output*): la función `import_previous_work` enlaza sus checkpoints, sus predicciones y sus resultados. Si no se añade, el cuaderno entrena los baselines con exactamente la misma receta, lo que suma unos 83 minutos (20 de SSD y 63 de Cascade R-CNN). El mismo mecanismo sirve para repartir las mejoras en varias sesiones: cada experimento guarda su checkpoint y sus predicciones, se salta si ya está hecho y los bloques grandes se pueden apagar con el diccionario `RUN`. Los experimentos de solo inferencia (TTA y ensemble) se ejecutan justo después de los baselines porque son baratos, de modo que ya hay resultados útiles aunque la sesión se corte durante los entrenamientos.

Todo lo que se genera queda en `/kaggle/working/work`: `results.csv` (todas las métricas de todos los experimentos), `ranking_mejoras.csv` (ranking observado frente al previsto), `delta_ap_por_clase_test.csv`, `ensemble_grid_val.csv` y las figuras en `figs/`.

### Reglas del estudio

Cada experimento cambia una sola cosa respecto a su referencia, que es el baseline del mismo detector, y no se combina ninguna mejora con otra. Las decisiones se toman con *val* y la cifra que se reporta es la de *test*. Todos los experimentos de entrenamiento usan la misma semilla, los mismos datos y los mismos criterios que el baseline, incluida la elección de la época de menor pérdida en validación; el mAP por época en *val* solo se registra para dibujar las curvas de aprendizaje.

Como cada configuración se entrena una sola vez, la diferencia entre dos experimentos puede deberse al azar. Para medir ese ruido se entrena un control: el baseline de SSD con otra semilla (`S0`). Una mejora solo se considera clara si supera esa diferencia tanto en *val* como en *test*.

### Qué se ha aprovechado del otro cuaderno y qué no

Del otro cuaderno se han tomado las seis propuestas, adaptadas para que cumplan las reglas anteriores y para que sean válidas con este baseline. El aumento de datos de SSD (E1) aplicaba cuatro transformaciones a la vez, así que se divide en cuatro experimentos independientes: volteo horizontal (S1), volteo vertical (S2), rotación de 90° (S3) y variación de color (S4). Su justificación, que las imágenes son cenitales, no se sostiene con las fotos del dataset, que muestran una perspectiva de ojo de pez con orientación clara, y por eso tratamos los volteos y las rotaciones como hipótesis que hay que comprobar y no como transformaciones seguras. El learning rate diferenciado (E2) se mantiene tal cual como S5.

La combinación de E1 y E2 (E3) se descarta porque mezcla varias mejoras en un mismo experimento; queda como siguiente paso una vez que se sepa cuáles funcionan por separado. El aumento de datos de Cascade R-CNN (E4) también se reduce: en este baseline Detectron2 ya aplica reescalado multiescala (640–800 px) y volteo horizontal, como confirma el registro de entrenamiento, de modo que lo único nuevo que se puede probar es el volteo vertical (C2) y el color (C3). El TTA (E5) y el ensemble (E6) se aplican a los baselines y no a «los mejores modelos», para que su efecto no se mezcle con el de otra mejora; el TTA se prueba con la versión de dos vistas (S6) y con la de tres, que incluye el volteo vertical (S7). Por último, la justificación del otro cuaderno para su baseline de Cascade (escalado lineal del learning rate de Detectron2) es en realidad una mejora de hiperparámetros respecto al nuestro, que usa un valor diez veces menor, y se incorpora como C1.

No se han incorporado otras diferencias de su baseline (calentamiento del LR, caída del LR en las épocas 10 y 13, lote de 4, precisión mixta, elección del checkpoint por mAP) porque cada una cambiaría varias cosas del baseline a la vez o solo afecta al protocolo; son candidatas para una segunda ronda. Tampoco se adopta una advertencia suya sobre el `score_thresh` de SSD: en torchvision el valor que se pasa al construir el modelo sí se respeta (se ha comprobado en el código de `ssd300_vgg16`).

## 1. Imports y configuración del entorno

In [ ]:
# Detectron2 se compila desde el repositorio y se omite si ya está instalado
!python -c "import detectron2" 2>/dev/null || pip -q install 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation
!pip -q install ensemble-boxes

In [ ]:
import os, sys, io, json, time, math, copy, random, tarfile, contextlib, gc, importlib.util
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from datasets import load_dataset

import torch
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Importamos las librerías de pycocotools para la evaluación del modelo y el manejo del dataset COCO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Importamos las librerías de Detectron2 para la configuración, entrenamiento y evaluación del modelo
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.evaluation import COCOEvaluator
from detectron2.data import (DatasetCatalog, MetadataCatalog, DatasetMapper,
                                build_detection_train_loader, transforms as T)
from detectron2.data.datasets import register_coco_instances

In [ ]:
SEED = 2710
DATASET_NAME = "lettuce-pallets"
HF_DATASET = "Francesco/lettuce-pallets" # solo se usa si no hay dataset en el input de kaggle

IN_KAGGLE = Path("/kaggle/working").exists()
KAGGLE_INPUT = Path("/kaggle/input")
ROOT_DIR = Path("/kaggle/working") if IN_KAGGLE else Path(".") # usamos el directorio actual en local
WORK_DIR = ROOT_DIR / "work" # checkpoints, predicciones, figuras y results.csv
DATA_DIR = ROOT_DIR / "data" # datos extraídos y limpios

PRETRAINED = True # pesos preentrenados en COCO

# Parámetros de inferencia comunes a ambos detectores
SCORE_THRESH_EVAL = 0.001 # umbral de confianza para filtrar predicciones durante la evaluación
NMS_IOU = 0.5 # umbral de IoU para la supresión de no-máximos durante la evaluación
MAX_DETS = 300 # número máximo de predicciones por imagen durante la evaluación

SPLITS = ("train", "val", "test")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| Kaggle:", IN_KAGGLE)

In [ ]:
import shutil, warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------------------------------------------
# Qué se ejecuta en esta sesión. Cada bloque se puede apagar para repartir el trabajo en varias sesiones
# de Kaggle (los checkpoints y las predicciones ya hechos se reutilizan; ver «import_previous_work»).
# Tiempo aproximado en una GPU como la de la primera ejecución: SSD ≈ 20-23 min por experimento,
# Cascade R-CNN ≈ 63 min por experimento, TTA y ensemble ≈ 5-10 min en total.
# ---------------------------------------------------------------------------------------------
RUN = {"control_seed": True,     # SSD baseline con otra semilla: mide el ruido entre ejecuciones (≈ 22 min)
       "ssd_aug":      True,     # S1 hflip, S2 vflip, S3 rot90, S4 color (≈ 4 × 22 min)
       "ssd_lr":       True,     # S5 LR del backbone × 0.1 (≈ 22 min)
       "cascade":      True,     # C1 LR × 10, C2 vflip, C3 color (≈ 3 × 63 min)
       "inferencia":   True}     # S6/S7 TTA y E1 ensemble (solo inferencia, sin entrenar)
CASCADE_EXPERIMENTS = ["C1", "C2", "C3"]   # quita alguno para ahorrar tiempo
PHOTO = dict(brightness=0.25, contrast=0.25, saturation=0.25)   # jitter fotométrico común a SSD y Cascade (±25 %)

In [ ]:
# Fijamos todas las semillas para asegurar la reproducibilidad de los experimentos
def seed_everything(seed=SEED, deterministic=True):
    """Fija todas las semillas. 'deterministic' hace cuDNN determinista (algo más lento)."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Función para fijar la semilla de los workers del DataLoader, si no el aumento de datos no es reproducible
def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

# Creamos un generador de números aleatorios para el DataLoader, que se utilizará para inicializar los workers y garantizar la reproducibilidad
def make_generator(seed=SEED):
    return torch.Generator().manual_seed(seed)

seed_everything() # Fijamos la semilla

# Creamos los subdirectorios necesarios para almacenar los resultados del entrenamiento y la evaluación
for sub in ("ckpt", "preds", "figs", "d2"):
    (WORK_DIR / sub).mkdir(parents=True, exist_ok=True)

# Creamos los subdirectorios necesarios para almacenar los datos limpios y procesados del dataset
DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR = DATA_DIR / "clean" # directorio para almacenar los datos limpios y procesados del dataset
CLEAN_DIR.mkdir(exist_ok=True)
print("WORK_DIR:", WORK_DIR.resolve())

In [ ]:
# Semilla de los workers del DataLoader. Se parte de torch.initial_seed(), que cambia en cada época pero sigue siendo
# reproducible (PyTorch lo deriva del generador del DataLoader); con una semilla fija por worker, el aumento de datos
# repetiría la misma secuencia de números aleatorios en todas las épocas.
def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

## 2. Datos: extracción y limpieza (idénticas al cuaderno de baselines)

In [ ]:
SPLIT_DIR_ALIASES = {"train": "train", "valid": "val", "validation": "val", "val": "val", "test": "test"}

# Extraemos el tar.gz si el destino está vacío
def extract_tar(tar_path, dest):
    """Extrae el tar.gz si el destino está vacío."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    if not any(dest.iterdir()):
        with tarfile.open(tar_path, "r:*") as tar:
            try: tar.extractall(dest, filter="data")
            except TypeError: tar.extractall(dest)
    return dest

# Localizamos los splits de entrenamiento, validación y test en el dataset extraído
def locate_raw_splits(root):
    raw = {}
    for js in Path(root).rglob("*.json"):
        key = SPLIT_DIR_ALIASES.get(js.parent.name.lower())
        if key and key not in raw:
            raw[key] = {"json": js, "images": js.parent} # en Roboflow, las imágenes están junto al JSON
    missing = [s for s in SPLITS if s not in raw]
    assert not missing, f"Faltan splits: {missing}"
    return raw

def find_tar_in_input():
    if not KAGGLE_INPUT.exists(): return None
    for pat in ("*.tar.gz", "*.tgz", "*.tar"):
        found = sorted(KAGGLE_INPUT.rglob(pat))
        if found: return found[0]
    return None

def export_hf(dest):
    """Descarga el dataset de Hugging Face y lo escribe como Roboflow: <split>/{imágenes, _annotations.coco.json}.
    Se conservan TODAS las categorías originales (incluida 'lettuce', id 0) para que build_clean_coco haga el mismo trabajo."""

    #Descargamos en memoria el dataset de Hugging Face y lo escribimos en el directorio de destino como Roboflow: <split>/{imágenes, _annotations.coco.json}.
    hf_ds = load_dataset(HF_DATASET)
    # Mapeamos los nombres internos de los splits con  los nombres de directorio que queremos usar en el destino
    dir_names = {"train": "train", "val": "valid", "test": "test"}
    # Accedemos al primer split disponible e inspeccionamos el esquema (features) de categoría de objects para tener los nombres de las clases
    feat = hf_ds[next(iter(hf_ds.keys()))].features["objects"]["category"]
    # En caso de que la columna de categoríais esté eun un contenedor de tipo Sequence/List, obtenemos el feature ClassLabel subyacente
    feat = getattr(feat, "feature", feat)                       # Sequence/List(ClassLabel) -> ClassLabel
    # Si el feature es de tipo ClassLabel, obtenemos los nombres de las clases; si no, dejamos names como None
    names = list(feat.names) if getattr(feat, "names", None) else None
    # Inicializamos un conjunto para almacenar las categorías observadas en el dataset
    observed = set()

    # Recorremos cada split del dataset
    for split_name, ds in hf_ds.items():
        # Normalizamos el nombre del split a minúsculas y determinamos la clave correspondiente en dir_names
        s = split_name.lower()
        key = "train" if s == "train" else "val" if "val" in s else "test" if "test" in s else None
        if key is None: continue
        out = Path(dest) / dir_names[key]
        out.mkdir(parents=True, exist_ok=True)
        # Inicializamos listas para almacenar la información de las imágenes y anotaciones
        images, anns = [], []
        # Iteramos sobre cada fila del split del dataset convirtiendo las imágenes a RGB y guardándolas en el directorio correspondiente, mientras recopilamos la información de las anotaciones
        for idx, row in enumerate(ds):
            img = row["image"].convert("RGB")
            fn = f"{idx:05d}.jpg"; img.save(out / fn, quality=95)
            images.append({"id": idx, "file_name": fn, "width": img.width, "height": img.height})
            objs = row["objects"]
            # Emparejamos cada bounding box con su categoría correspondiente y agregamos la información a la lista de anotaciones, mientras registramos las categorías observadas
            for bbox, cat in zip(objs["bbox"], objs["category"]):
                x, y, w, h = map(float, bbox); observed.add(int(cat))
                anns.append({"id": len(anns) + 1, "image_id": idx, "category_id": int(cat),
                             "bbox": [x, y, w, h], "area": w * h, "iscrowd": 0})
        # Escribimos el archivo _annotations.coco.json para el split actual, incluyendo las imágenes y anotaciones recopiladas
        (out / "_annotations.coco.json").write_text(json.dumps({"images": images, "annotations": anns, "categories": []}))
    # Actualizamos el archivo _annotations.coco.json para cada split, incluyendo las categorías
    cat_ids = range(len(names)) if names else sorted(observed)
    cats = [{"id": i, "name": names[i] if names else str(i)} for i in cat_ids]
    for d in dir_names.values():
        p = Path(dest) / d / "_annotations.coco.json"
        j = json.loads(p.read_text()); j["categories"] = cats; p.write_text(json.dumps(j))
    return dest

raw_root = DATA_DIR / "raw"
if not raw_root.exists() or not any(raw_root.iterdir()):
    print("Preparando datos RAW...")
    RAW = locate_raw_splits(export_hf(raw_root))
else:
    RAW = locate_raw_splits(raw_root)   # sesión con datos ya extraídos
for s in SPLITS: print(f"{s}: {RAW[s]['json']}")

In [ ]:
def build_clean_coco(raw, out_dir):
    """Genera un JSON COCO limpio por split. Devuelve (CLEAN, class_names, class_ids)."""
    # Creamos el directorio de salida si no existe
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Leemos el JSON de entrenamiento original y obtenemos los nombres de las categorías y los IDs utilizados
    train = json.loads(Path(raw["train"]["json"]).read_text())
    old_names = {c["id"]: c["name"] for c in train["categories"]}
    used_ids = sorted({a["category_id"] for a in train["annotations"]})
    dropped = [n for i, n in old_names.items() if i not in used_ids] # categorías descartadas
    print("Categorías descartadas (sin anotaciones en train):", dropped)

    # Creamos un diccionario que mapea los nombres de las categorías a nuevos IDs consecutivos (1..K) para las categorías utilizadas
    name2new = {old_names[i]: n + 1 for n, i in enumerate(used_ids)}

    # Iteramos sobre cada split y generamos un nuevo JSON COCO limpio
    # Descartamos anotaciones de categorías no utilizadas y cajas degeneradas
    clean = {}
    for split in SPLITS:
        d = json.loads(Path(raw[split]["json"]).read_text())
        id2name = {c["id"]: c["name"] for c in d["categories"]}
        imgs = {im["id"]: im for im in d["images"]}
        new_anns, stats = [], Counter()
        
        for a in d["annotations"]:
            name = id2name.get(a["category_id"])
            
            # Descartamos cajas degeneradas (cajas con ancho o alto menor que 1 píxel)
            im = imgs[a["image_id"]]; x, y, w, h = a["bbox"]
            x1, y1 = max(0.0, x), max(0.0, y)
            x2, y2 = min(float(im["width"]), x + w), min(float(im["height"]), y + h)
            if x2 - x1 < 1 or y2 - y1 < 1:
                stats["caja_degenerada"] += 1
                continue
            
            # Creamos la nueva anotación con el nuevo ID de categoría y la caja ajustada
            new_anns.append({"id": len(new_anns) + 1, "image_id": a["image_id"],
                             "category_id": name2new[name], "bbox": [x1, y1, x2 - x1, y2 - y1],
                             "area": (x2 - x1) * (y2 - y1), "iscrowd": 0})
        
        # Guardamos el nuevo JSON COCO limpio para el split actual
        out = {"images": d["images"], "annotations": new_anns,
               "categories": [{"id": i, "name": n} for n, i in name2new.items()]}
        
        # Guardamos el JSON limpio en el directorio de salida y almacenamos la información del split en el diccionario 'clean'
        p = out_dir / f"{split}_clean.json"; p.write_text(json.dumps(out))
        clean[split] = {"json": p, "images": Path(raw[split]["images"])}
        print(f"{split:5s}: {len(d['images'])} imgs | {len(d['annotations'])} -> {len(new_anns)} anotaciones | descartes: {dict(stats)}")
    names = list(name2new.keys())
    ids = list(name2new.values())
    return clean, names, ids

CLEAN, CLASS_NAMES, CLASS_IDS = build_clean_coco(RAW, CLEAN_DIR)
K = len(CLASS_NAMES) # nº de clases reales
# Creamos un diccionario que mapea los IDs de las clases a sus índices
CID2IDX = {c: i for i, c in enumerate(CLASS_IDS)}
print(f"\nK = {K} clases:", dict(zip(CLASS_IDS, CLASS_NAMES)))

# Creamos un diccionario que mapea los splits a sus objetos COCO correspondientes para facilitar la evaluación
COCO_GT = {s: COCO(str(CLEAN[s]["json"])) for s in SPLITS}

## 3. Infraestructura común

La evaluación (`evaluate_coco`), el informe (`print_metrics`, `evaluate_and_report`), el registro en `results.csv` y la medida de tiempos (`time_inference`) son los del cuaderno de baselines, sin cambios, para que los números de los experimentos sean comparables con los de los baselines. A continuación se importa el trabajo previo, si lo hay, y se definen el dataset con aumento opcional y las funciones de entrenamiento.

El dataset de entrenamiento de SSD admite ahora una transformación que modifica la imagen y las cajas a la vez. Cada operación tiene su propia probabilidad, de forma que cada experimento activa solo una: volteo horizontal, volteo vertical, rotación de 90° (con la fórmula de rotación de cajas comprobada contra `rot90` de NumPy) o variación de brillo, contraste y saturación de ±25 %. Val y test no se aumentan nunca. La semilla de los workers del DataLoader se deriva de `torch.initial_seed()`, que cambia en cada época pero sigue siendo reproducible; con una semilla fija el aumento repetiría la misma secuencia aleatoria en todas las épocas.

In [ ]:
METRIC_NAMES = ["mAP", "mAP50", "mAP75", "mAP_small", "mAP_medium", "mAP_large"]

def run_cocoeval(coco_gt, predictions, max_dets=MAX_DETS):
    """Ejecuta COCOeval (bbox) sin imprimir nada y devuelve el objeto con los tensores de precisión y recall."""
    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(predictions)
        ev = COCOeval(coco_gt, coco_dt, "bbox")
        ev.params.maxDets = [1, 10, max_dets]
        ev.evaluate(); ev.accumulate()
    return ev

def _mean_valid(x):
    """Media de los valores válidos (COCOeval marca con -1 lo que no se puede calcular)."""
    x = np.asarray(x); x = x[x > -1]
    return float(x.mean()) if x.size else float("nan")

def evaluate_coco(coco_gt, predictions, max_dets=MAX_DETS):
    """Evaluación COCO (bbox) idéntica para SSD y Cascade R-CNN.
    Las métricas se calculan directamente sobre el tensor de precisión [T, R, K, A, M]. No usamos ev.stats[0]
    porque COCOeval.summarize() fija maxDets=100 para el mAP@[.5:.95] y, al haber definido maxDets=[1, 10, 300],
    ese valor salía como -1."""
    if not predictions:
        return {**{n: 0.0 for n in METRIC_NAMES}, **{f"AP_{c}": 0.0 for c in CLASS_NAMES}}
    ev = run_cocoeval(coco_gt, predictions, max_dets)
    prec = ev.eval["precision"][..., 2]               # [T, R, K, A]; M=2 -> max_dets
    iou = ev.params.iouThrs
    t50, t75 = int(np.argmin(np.abs(iou - 0.50))), int(np.argmin(np.abs(iou - 0.75)))
    m = {"mAP":        _mean_valid(prec[:, :, :, 0]),   # A=0: todas las áreas
         "mAP50":      _mean_valid(prec[t50, :, :, 0]),
         "mAP75":      _mean_valid(prec[t75, :, :, 0]),
         "mAP_small":  _mean_valid(prec[:, :, :, 1]),
         "mAP_medium": _mean_valid(prec[:, :, :, 2]),
         "mAP_large":  _mean_valid(prec[:, :, :, 3])}
    for k, cname in enumerate(CLASS_NAMES):           # los catIds de COCOeval van ordenados = CLASS_IDS
        m[f"AP_{cname}"] = _mean_valid(prec[:, :, k, 0])
    return m

RESULTS_CSV = WORK_DIR / "results.csv"

def log_result(exp, detector, split, metrics, train_time_s=float("nan"), infer_ms_img=float("nan"), params_m=float("nan")):
    """Añade (o sustituye) una fila en results.csv y devuelve la tabla completa."""
    row = {"exp": exp, "detector": detector, "split": split, **metrics,
           "train_min": round(train_time_s / 60, 1), "infer_ms_img": round(infer_ms_img, 1),
           "params_M": round(params_m, 1)}
    df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
    if len(df):
        df = df[~((df["exp"] == exp) & (df["split"] == split))]
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])
    df.to_csv(RESULTS_CSV, index=False)
    return df

def sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()

def count_params(model):
    """Número de parámetros (en millones)."""
    return sum(p.numel() for p in model.parameters()) / 1e6

@torch.no_grad()
def time_inference(infer_fn, coco_gt, image_dir, n=100, warmup=10):
    """Tiempo medio de inferencia (ms/imagen) con el MISMO protocolo para cualquier detector:
    batch de 1 imagen, imágenes ya cargadas en memoria (no se mide la lectura de disco), calentamiento previo
    y sincronización de la GPU. Incluye el preprocesado, la red y el post-procesado (NMS) propios de cada detector.
    infer_fn recibe una imagen RGB uint8 (H, W, 3) como array de NumPy."""
    ids = sorted(coco_gt.imgs)[: n + warmup]
    imgs = [np.array(Image.open(Path(image_dir) / coco_gt.imgs[i]["file_name"]).convert("RGB")) for i in ids]
    for im in imgs[:warmup]:
        infer_fn(im)
    sync(); t0 = time.perf_counter()
    for im in imgs[warmup:]:
        infer_fn(im)
    sync()
    return 1000 * (time.perf_counter() - t0) / len(imgs[warmup:])

def print_metrics(exp, split, m, infer_ms=float("nan"), train_s=float("nan"), params_m=float("nan"), coco_gt=None):
    """Informe con el mismo formato para todos los detectores y experimentos."""
    line = "═" * 86
    n_small = sum(a["area"] < 32 ** 2 for a in coco_gt.anns.values()) if coco_gt is not None else None
    print(line); print(f" {exp}   |   split: {split}"); print(line)
    print(f" mAP@[.5:.95] = {m['mAP']:.3f}    mAP@.50 = {m['mAP50']:.3f}    mAP@.75 = {m['mAP75']:.3f}")
    nota = f"   (solo {n_small} objetos small en el GT: poco fiable)" if n_small is not None and n_small < 30 else ""
    print(f" Por tamaño:   small = {m['mAP_small']:.3f}    medium = {m['mAP_medium']:.3f}    large = {m['mAP_large']:.3f}{nota}")
    print(" AP por clase: " + "    ".join(f"{c} = {m[f'AP_{c}']:.3f}" for c in CLASS_NAMES))
    print(f" Coste:        {infer_ms:.1f} ms/img ({1000 / infer_ms:.1f} FPS)    entrenamiento = {train_s / 60:.1f} min    "
          f"{params_m:.1f} M parámetros\n")

def evaluate_and_report(tag, exp, detector, split, preds, infer_ms, train_s, params_m):
    """Guarda las predicciones, evalúa, registra en results.csv e imprime el informe común."""
    (WORK_DIR / "preds" / f"{tag}_{split}.json").write_text(json.dumps(preds))
    m = evaluate_coco(COCO_GT[split], preds)
    log_result(exp, detector, split, m, train_s, infer_ms, params_m)
    print_metrics(exp, split, m, infer_ms, train_s, params_m, COCO_GT[split])
    return m

In [ ]:
def import_previous_work(max_copy_mb=50, max_depth=5):
    """Reutiliza el trabajo de sesiones anteriores. Si añades como *Input* la salida de otra sesión de Kaggle (la del cuaderno de
    baselines o la de una sesión anterior de este cuaderno), se enlazan sus checkpoints y predicciones y se fusiona su results.csv.
    Los ficheros pequeños se copian y los grandes (modelos de Cascade, ≈ 0,5 GB) se enlazan sin copiarlos."""
    if not KAGGLE_INPUT.exists(): return
    works = []
    for root, dirs, _ in os.walk(KAGGLE_INPUT):
        if len(Path(root).relative_to(KAGGLE_INPUT).parts) >= max_depth: dirs[:] = []
        if "work" in dirs: works.append(Path(root) / "work"); dirs.remove("work")
    n_files = n_rows = 0
    for src in works:
        for f in src.rglob("*"):
            if not f.is_file() or f.name == "results.csv" or f.suffix not in (".pth", ".json", ".yaml"): continue
            dest = WORK_DIR / f.relative_to(src)
            if dest.exists() or dest.is_symlink(): continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            if f.stat().st_size < max_copy_mb * 2**20: shutil.copy(f, dest)
            else: dest.symlink_to(f)
            n_files += 1
        csv = src / "results.csv"
        if csv.exists():
            imp = pd.read_csv(csv)
            loc = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(columns=imp.columns)
            have = set(zip(loc["exp"], loc["split"]))
            new = imp[[(e, s) not in have for e, s in zip(imp["exp"], imp["split"])]]
            if len(new): pd.concat([loc, new], ignore_index=True).to_csv(RESULTS_CSV, index=False); n_rows += len(new)
    print(f"Trabajo previo importado desde {len(works)} carpeta(s): {n_files} ficheros, {n_rows} filas de resultados.")

import_previous_work()

In [ ]:
assert CLASS_IDS == list(range(1, K + 1)), "Los ids deben ser 1..K para que label == category_id"

def rot90_boxes(boxes, W, H, k):
    """Rota k veces 90° en sentido antihorario (igual que torch.rot90(img, k, dims=(1, 2))). boxes: [N,4] xyxy."""
    for _ in range(k % 4):
        x1, y1, x2, y2 = boxes.unbind(-1)
        boxes = torch.stack([y1, W - x2, y2, W - x1], dim=-1)    # nuevo_x = y ; nuevo_y = W - x
        W, H = H, W
    return boxes, W, H

class DetAugment:
    """Aumento de datos que transforma la imagen y las cajas a la vez. Cada operación tiene su propia probabilidad
    (0 la desactiva), de modo que cada experimento activa UNA sola: hflip, vflip, rot90 o color (jitter fotométrico)."""
    def __init__(self, hflip=0.0, vflip=0.0, rot90=0.0, color=0.0, brightness=0.25, contrast=0.25, saturation=0.25):
        self.hflip, self.vflip, self.rot90, self.color = hflip, vflip, rot90, color
        self.b, self.c, self.s = brightness, contrast, saturation

    def __call__(self, img, target):
        _, H, W = img.shape; boxes = target["boxes"].clone()
        if random.random() < self.hflip:
            img = img.flip(-1); boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
        if random.random() < self.vflip:
            img = img.flip(-2); boxes[:, [1, 3]] = H - boxes[:, [3, 1]]
        if random.random() < self.rot90:
            k = random.choice([1, 2, 3])
            img = torch.rot90(img, k, dims=(1, 2)); boxes, W, H = rot90_boxes(boxes, W, H, k)
        if random.random() < self.color:
            img = TF.adjust_brightness(img, random.uniform(1 - self.b, 1 + self.b))
            img = TF.adjust_contrast(img, random.uniform(1 - self.c, 1 + self.c))
            img = TF.adjust_saturation(img, random.uniform(1 - self.s, 1 + self.s)).clamp(0, 1)
        target = dict(target); target["boxes"] = boxes
        return img, target

class CocoDetection(Dataset):
    """Dataset COCO para torchvision. Devuelve (imagen [0,1] CxHxW, target con cajas xyxy y etiquetas 1..K).
    Igual que en el cuaderno de baselines, pero con aumento opcional (solo se pasa al split de entrenamiento)."""
    def __init__(self, coco, image_dir, augment=None):
        self.coco, self.image_dir, self.augment = coco, Path(image_dir), augment
        self.ids = sorted(coco.getImgIds())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        img_id = self.ids[index]; info = self.coco.imgs[img_id]
        image = TF.to_tensor(Image.open(self.image_dir / info["file_name"]).convert("RGB"))
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
        boxes = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]] for a in anns]
        target = {"boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),     # reshape: imágenes sin objetos
                  "labels": torch.as_tensor([a["category_id"] for a in anns], dtype=torch.int64),
                  "image_id": torch.tensor([img_id])}
        if self.augment is not None:
            image, target = self.augment(image, target)
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

BATCH_SSD, NUM_WORKERS = 4, 2
# val y test NUNCA se aumentan
DS = {s: CocoDetection(COCO_GT[s], CLEAN[s]["images"]) for s in SPLITS}
CLASS_COLORS = plt.cm.tab10(np.arange(K))
print({s: len(DS[s]) for s in SPLITS})

### SSD300 y Cascade R-CNN con un solo cambio

`run_ssd_experiment` reproduce la receta del baseline (SGD con momentum 0,9, learning rate 0,002, decaimiento por pasos cada 5 épocas, 15 épocas, recorte de gradiente 10 y selección de la época de menor pérdida en validación) y solo deja libres el aumento de datos y el factor del learning rate del backbone. Con los valores por defecto es exactamente el baseline, de modo que el baseline se reentrena con la misma función si no se encuentra su checkpoint. `run_cascade_experiment` hace lo mismo con la configuración de Detectron2 del baseline: lo único que admite cambiar es el learning rate base, el volteo vertical o la variación de color. La saturación se implementa a medida porque Detectron2 lee las imágenes en BGR y su transformación de saturación supone RGB.

In [ ]:
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead

def build_ssd(pretrained=True):
    """Idéntico al baseline: SSD300-VGG16 con pesos COCO y cabeza de clasificación nueva para K clases (+1 fondo)."""
    weights = SSD300_VGG16_Weights.COCO_V1 if pretrained else None
    model = ssd300_vgg16(weights=weights, weights_backbone=None, score_thresh=SCORE_THRESH_EVAL,
                         nms_thresh=NMS_IOU, detections_per_img=MAX_DETS)
    in_channels = [layer.in_channels for layer in model.head.classification_head.module_list]
    num_anchors = model.anchor_generator.num_anchors_per_location()
    model.head.classification_head = SSDClassificationHead(in_channels=in_channels, num_anchors=num_anchors, num_classes=K + 1)
    return model

def train_one_epoch(model, loader, optimizer, device):
    model.train(); losses = []
    for images, targets in loader:
        images = [im.to(device) for im in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss = sum(model(images, targets).values())
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step(); losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def mean_detection_loss(model, loader, device):
    model.train(); vals = []                      # torchvision solo calcula la pérdida en modo train (sin actualizar pesos)
    for images, targets in loader:
        images = [im.to(device) for im in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        vals.append(float(sum(model(images, targets).values()).cpu()))
    model.eval()
    return float(np.mean(vals))

@torch.no_grad()
def predict_ssd(model, dataset, batch_size=BATCH_SSD, flip=None):
    """Predicciones en formato COCO. flip in {None, 'h', 'v'}: predice sobre la imagen volteada y deshace el volteo en las cajas (TTA)."""
    model.eval(); preds = []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS)
    for images, targets in loader:
        imgs = [im.to(DEVICE) for im in images]
        if flip == "h": imgs = [im.flip(-1) for im in imgs]
        elif flip == "v": imgs = [im.flip(-2) for im in imgs]
        for im, target, out in zip(images, targets, model(imgs)):
            H, W = im.shape[-2:]
            boxes = out["boxes"].cpu().numpy().copy()
            if flip == "h": boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
            elif flip == "v": boxes[:, [1, 3]] = H - boxes[:, [3, 1]]
            for (x1, y1, x2, y2), s, l in zip(boxes, out["scores"].cpu().tolist(), out["labels"].cpu().tolist()):
                preds.append({"image_id": int(target["image_id"].item()), "category_id": CLASS_IDS[int(l) - 1],
                              "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(s)})
    return preds

EPOCHS_SSD, LR_SSD = 15, 0.002

def ssd_paths(tag):
    return WORK_DIR / "ckpt" / f"{tag}.pth", WORK_DIR / "ckpt" / f"{tag}_history.json"

def ssd_train_time(tag):
    return json.loads(ssd_paths(tag)[1].read_text())["train_time_s"]

def is_done(tag, exp):
    """¿Están ya las predicciones y los resultados de val y test de este experimento?"""
    if not all((WORK_DIR / "preds" / f"{tag}_{s}.json").exists() for s in ("val", "test")) or not RESULTS_CSV.exists(): return False
    r = pd.read_csv(RESULTS_CSV); return {"val", "test"} <= set(r[r.exp == exp]["split"])

def run_ssd_experiment(tag, exp, augment=None, backbone_lr_mult=1.0, seed=SEED, reevaluate=False):
    """Entrena (o reutiliza) un SSD300 con EXACTAMENTE la receta del baseline y UN solo cambio (augment o backbone_lr_mult),
    se queda con la época de menor pérdida en val (mismo criterio que el baseline), evalúa val y test y lo registra.
    Además guarda por época el mAP@[.5:.95] en val, solo para dibujar la curva de aprendizaje (no interviene en la selección)."""
    ckpt, hist_path = ssd_paths(tag)
    if is_done(tag, exp) and not reevaluate:
        print(f"[{tag}] ya hecho (checkpoint, predicciones y resultados): se omite"); return
    seed_everything(seed)
    model = build_ssd(PRETRAINED).to(DEVICE)
    if not (ckpt.exists() and hist_path.exists()):
        ds_train = CocoDetection(COCO_GT["train"], CLEAN["train"]["images"], augment=DetAugment(**augment, **PHOTO) if augment else None)
        train_loader = DataLoader(ds_train, batch_size=BATCH_SSD, shuffle=True, collate_fn=collate_fn, num_workers=NUM_WORKERS,
                                  worker_init_fn=seed_worker, generator=make_generator(seed))
        val_loader = DataLoader(DS["val"], batch_size=BATCH_SSD, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS)
        if backbone_lr_mult == 1.0:
            groups = [p for p in model.parameters() if p.requires_grad]                  # un solo grupo, como el baseline
        else:                                                                             # LR diferenciado: backbone más lento que la cabeza
            bb = list(model.backbone.parameters()); bb_ids = {id(p) for p in bb}
            groups = [{"params": [p for p in model.parameters() if id(p) not in bb_ids], "lr": LR_SSD},
                      {"params": bb, "lr": LR_SSD * backbone_lr_mult}]
        optimizer = torch.optim.SGD(groups, lr=LR_SSD, momentum=0.9, weight_decay=0.0005)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
        history, best_val = [], float("inf"); sync(); t0 = time.time()
        for epoch in range(EPOCHS_SSD):
            tr = train_one_epoch(model, train_loader, optimizer, DEVICE)
            vl = mean_detection_loss(model, val_loader, DEVICE); scheduler.step()
            m = evaluate_coco(COCO_GT["val"], predict_ssd(model, DS["val"]))
            history.append({"epoch": epoch + 1, "train_loss": tr, "val_loss": vl, "val_mAP": m["mAP"], "val_mAP50": m["mAP50"]})
            print(f"[{tag}] epoch {epoch+1}/{EPOCHS_SSD}: train={tr:.4f} val={vl:.4f} | val mAP={m['mAP']:.3f} mAP50={m['mAP50']:.3f}")
            if vl < best_val: best_val = vl; torch.save(model.state_dict(), ckpt)
        sync(); hist_path.write_text(json.dumps({"history": history, "train_time_s": time.time() - t0}))
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); model.eval()
    ms = time_inference(lambda im: model([TF.to_tensor(im).to(DEVICE)]), COCO_GT["test"], CLEAN["test"]["images"])
    for split in ("val", "test"):
        evaluate_and_report(tag, exp, "SSD300", split, predict_ssd(model, DS[split]), ms, ssd_train_time(tag), count_params(model))
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from fvcore.transforms.transform import BlendTransform

# Registro de los splits limpios en Detectron2
for split in SPLITS:
    name = f"lettuce_{split}"
    if name in DatasetCatalog.list(): DatasetCatalog.remove(name)
    register_coco_instances(name, {}, str(CLEAN[split]["json"]), str(CLEAN[split]["images"]))
    MetadataCatalog.get(name).set(thing_classes=CLASS_NAMES)

CASCADE_CONFIG = "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"

def make_cascade_cfg(output_dir, base_lr=0.00025):
    """Configuración idéntica a la del baseline; solo base_lr es un parámetro (experimento C1)."""
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(CASCADE_CONFIG))
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(CASCADE_CONFIG) if PRETRAINED else ""
    cfg.MODEL.MASK_ON = False
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = K
    cfg.DATASETS.TRAIN = ("lettuce_train",)
    cfg.DATASETS.TEST = ("lettuce_val",)
    cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS
    cfg.SEED = SEED
    cfg.SOLVER.IMS_PER_BATCH = 2
    cfg.SOLVER.BASE_LR = base_lr
    cfg.SOLVER.MAX_ITER = max(500, len(DS["train"]) // 2 * 10)
    cfg.SOLVER.STEPS = (int(cfg.SOLVER.MAX_ITER * 0.7), int(cfg.SOLVER.MAX_ITER * 0.9))
    cfg.SOLVER.CHECKPOINT_PERIOD = max(100, cfg.SOLVER.MAX_ITER // 3)
    cfg.TEST.EVAL_PERIOD = max(100, cfg.SOLVER.MAX_ITER // 5)
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_EVAL
    cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = NMS_IOU
    cfg.TEST.DETECTIONS_PER_IMAGE = MAX_DETS
    cfg.OUTPUT_DIR = str(output_dir)
    return cfg

class CascadeTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name):
        return COCOEvaluator(dataset_name, distributed=False, output_dir=os.path.join(cfg.OUTPUT_DIR, "eval"))

class RandomSaturationBGR(T.Augmentation):
    """Saturación aleatoria para imágenes en BGR. Detectron2 lee las imágenes en BGR y su RandomSaturation asume RGB,
    por lo que calcularía mal la imagen en gris; aquí se usan los pesos de luminancia en el orden de canales correcto."""
    def __init__(self, intensity_min, intensity_max):
        super().__init__(); self._init(locals())
    def get_transform(self, image):
        w = np.random.uniform(self.intensity_min, self.intensity_max)
        gray = image.dot([0.114, 0.587, 0.299])[:, :, np.newaxis]
        return BlendTransform(src_image=gray, src_weight=1 - w, dst_weight=w)

def make_trainer_class(vflip=False, color=False):
    """Trainer con las MISMAS transformaciones que el baseline (reescalado multiescala + volteo horizontal, que Detectron2
    aplica por defecto) más, como mucho, UNA adicional: volteo vertical o jitter fotométrico."""
    class Trainer(CascadeTrainer):
        @classmethod
        def build_train_loader(cls, cfg):
            augs = [T.ResizeShortestEdge(cfg.INPUT.MIN_SIZE_TRAIN, cfg.INPUT.MAX_SIZE_TRAIN, cfg.INPUT.MIN_SIZE_TRAIN_SAMPLING),
                    T.RandomFlip(prob=0.5, horizontal=True, vertical=False)]                      # = baseline
            if vflip: augs.append(T.RandomFlip(prob=0.5, horizontal=False, vertical=True))
            if color:
                lo, hi = 1 - PHOTO["brightness"], 1 + PHOTO["brightness"]
                augs += [T.RandomBrightness(lo, hi), T.RandomContrast(lo, hi), RandomSaturationBGR(lo, hi)]
            return build_detection_train_loader(cfg, mapper=DatasetMapper(cfg, is_train=True, augmentations=augs))
    return Trainer

def predict_d2(predictor, coco_gt, image_dir):
    preds = []
    for iid in sorted(coco_gt.imgs):
        img = cv2.imread(str(Path(image_dir) / coco_gt.imgs[iid]["file_name"]))   # BGR
        inst = predictor(img)["instances"].to("cpu")
        for (x1, y1, x2, y2), s, c in zip(inst.pred_boxes.tensor.numpy(), inst.scores.numpy(), inst.pred_classes.numpy()):
            preds.append({"image_id": int(iid), "category_id": CLASS_IDS[int(c)],
                          "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(s)})
    return preds

def cascade_paths(tag):
    d = WORK_DIR / "d2" / tag; d.mkdir(parents=True, exist_ok=True)
    return d, d / "model_final.pth", d / "train_time.json"

def cascade_train_time(tag):
    return json.loads(cascade_paths(tag)[2].read_text())["train_time_s"]

def get_cascade_predictor(tag, base_lr=0.00025):
    d, final, _ = cascade_paths(tag)
    cfg = make_cascade_cfg(d, base_lr); cfg.MODEL.WEIGHTS = str(final)
    return DefaultPredictor(cfg)

def run_cascade_experiment(tag, exp, base_lr=0.00025, vflip=False, color=False, reevaluate=False):
    """Entrena (o reutiliza) Cascade R-CNN con la receta del baseline y UN solo cambio (base_lr, vflip o color)."""
    d, final, tfile = cascade_paths(tag)
    if is_done(tag, exp) and not reevaluate:
        print(f"[{tag}] ya hecho (checkpoint, predicciones y resultados): se omite"); return
    cfg = make_cascade_cfg(d, base_lr)
    if not (final.exists() and tfile.exists()):
        seed_everything()
        trainer = make_trainer_class(vflip, color)(cfg)
        trainer.resume_or_load(resume=True)          # reanuda si la sesión se interrumpió
        sync(); t0 = time.time(); trainer.train(); sync()
        tfile.write_text(json.dumps({"train_time_s": time.time() - t0}))
        del trainer; gc.collect(); torch.cuda.empty_cache()
    cfg.MODEL.WEIGHTS = str(final); predictor = DefaultPredictor(cfg)
    ms = time_inference(lambda im: predictor(np.ascontiguousarray(im[:, :, ::-1])), COCO_GT["test"], CLEAN["test"]["images"])
    for split in ("val", "test"):
        evaluate_and_report(tag, exp, "Cascade R-CNN", split, predict_d2(predictor, COCO_GT[split], CLEAN[split]["images"]),
                            ms, cascade_train_time(tag), count_params(predictor.model))
    del predictor; gc.collect(); torch.cuda.empty_cache()

In [ ]:
def wbf_image(items_per_model, W, H, weights=None, iou_thr=0.55, skip_box_thr=0.001):
    """Weighted Boxes Fusion de las detecciones de UNA imagen (lista de listas de dicts COCO, una por modelo o por vista).
    Devuelve una lista de dicts COCO. WBF promedia las cajas que coinciden en lugar de descartar las de menor puntuación
    como hace el NMS, de modo que además de filtrar duplicados afina la localización."""
    from ensemble_boxes import weighted_boxes_fusion
    bl, sl, ll = [], [], []
    for items in items_per_model:
        b = np.array([[p["bbox"][0] / W, p["bbox"][1] / H, (p["bbox"][0] + p["bbox"][2]) / W, (p["bbox"][1] + p["bbox"][3]) / H]
                      for p in items], dtype=np.float32).reshape(-1, 4).clip(0, 1)
        bl.append(b); sl.append(np.array([p["score"] for p in items], dtype=np.float32))
        ll.append(np.array([p["category_id"] for p in items], dtype=np.float32))
    if sum(len(b) for b in bl) == 0: return []
    boxes, scores, labels = weighted_boxes_fusion(bl, sl, ll, weights=weights, iou_thr=iou_thr, skip_box_thr=skip_box_thr, conf_type="avg")
    out = []
    for b, s, l in zip(boxes, scores, labels):
        x1, y1, x2, y2 = b[0] * W, b[1] * H, b[2] * W, b[3] * H
        if x2 - x1 > 0 and y2 - y1 > 0:
            out.append({"category_id": int(l), "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(s)})
    return out

def fuse_predictions(coco_gt, pred_lists, **kw):
    """WBF de varias listas de predicciones COCO (distintos modelos para un ensemble, o distintas vistas del mismo modelo para TTA)."""
    per_model = []
    for preds in pred_lists:
        d = defaultdict(list)
        for p in preds: d[p["image_id"]].append(p)
        per_model.append(d)
    out = []
    for iid in coco_gt.getImgIds():
        W, H = coco_gt.imgs[iid]["width"], coco_gt.imgs[iid]["height"]
        for p in wbf_image([d.get(iid, []) for d in per_model], W, H, **kw): out.append({"image_id": int(iid), **p})
    return out

def load_preds(tag, split):
    return json.loads((WORK_DIR / "preds" / f"{tag}_{split}.json").read_text())

def ssd_items(out, W, H, flip=None):
    """Salida de SSD para una imagen -> lista de dicts COCO, deshaciendo el volteo (para medir el tiempo real de TTA y ensemble)."""
    boxes = out["boxes"].cpu().numpy().copy()
    if flip == "h": boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
    elif flip == "v": boxes[:, [1, 3]] = H - boxes[:, [3, 1]]
    return [{"category_id": CLASS_IDS[int(l) - 1], "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(s)}
            for (x1, y1, x2, y2), s, l in zip(boxes, out["scores"].cpu().tolist(), out["labels"].cpu().tolist())]

def d2_items(inst):
    inst = inst.to("cpu")
    return [{"category_id": CLASS_IDS[int(c)], "bbox": [float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])], "score": float(s)}
            for b, s, c in zip(inst.pred_boxes.tensor.numpy(), inst.scores.numpy(), inst.pred_classes.numpy())]

## 4. Predicción previa: qué mejoras esperamos que funcionen mejor

Antes de ejecutar ningún experimento fijamos qué esperamos de cada uno, para poder comprobar después si el razonamiento era correcto. La predicción se apoya en lo que mostraron los baselines. El mAP@.5 ya es de 0,94, mientras que el mAP@[.5:.95] es de 0,65–0,66, de modo que el margen de mejora está en la localización fina (la clase `Ready` tiene un AP de 0,53 a IoU 0,75 y su recall se queda en 0,72–0,78) y en las confusiones entre clases (`Ready` con `young` y `germination` con `pod`, además de muchos falsos positivos de `germination`). Además, SSD sobreajusta (la pérdida de validación deja de bajar en la época 8 con 0,91 en entrenamiento frente a 1,72 en validación) y Cascade R-CNN parece infraentrenado (su AP50 en validación seguía subiendo en la última evaluación, de 82,5 a 93,8, con una pérdida que no se había estabilizado).

Esperamos que el mayor beneficio sea para el **ensemble con WBF (E1)**, con una mejora de entre 1 y 3 puntos de mAP. Combina dos arquitecturas distintas con calidad parecida cuyos errores no coinciden (SSD confunde menos `young` con `Ready`, 39 casos frente a 89, y produce menos falsos positivos, 255 frente a 352, mientras que Cascade deja menos objetos sin detectar, 159 frente a 197), y WBF promedia las coordenadas de las cajas, lo que ataca justo el problema de localización fina. Su coste es sumar los tiempos de ambos detectores. En segundo lugar situamos el **learning rate de Cascade (C1)**, con una ganancia esperada de entre 0 y 3 puntos y la mayor incertidumbre: el valor actual es diez veces menor que el que prescribe el escalado lineal de Detectron2 y el calentamiento consume casi dos de las diez épocas, pero si el modelo ya estuviera cerca de su techo el cambio no serviría de nada.

Después esperamos las mejoras que atacan el sobreajuste de SSD. El **volteo horizontal (S1)** es la transformación más segura, duplica los datos efectivos y no cambia la distribución de formas de las cajas; esperamos entre 0,3 y 2 puntos. El **TTA con dos vistas (S6)** debería aportar entre 0,2 y 1,5 puntos por la misma razón que el ensemble, aunque con una sola arquitectura el error es más correlacionado. El **color (S4)** aporta algo porque la iluminación cambia mucho entre imágenes, pero val y test salen de la misma instalación, así que lo esperado es poco. El **learning rate del backbone (S5)** es incierto: puede regularizar y ayudar, pero también frenar la adaptación de un backbone entrenado con fotos muy distintas a este cultivo. El **color en Cascade (C3)** debería ser casi neutro porque el modelo ya parte de un entrenamiento con multiescala y volteo.

Esperamos que las tres últimas empeoren o no ayuden. El **TTA con volteo vertical (S7)** añade una vista fuera de distribución, porque el modelo nunca ha visto imágenes boca abajo, y debería quedar por debajo de S6. El **volteo vertical** (**C2** y **S2**) rompe la perspectiva de ojo de pez: la lámpara está en la parte superior de la imagen y el tamaño de los objetos depende de su altura en la foto, una pista que el modelo aprovecha. La **rotación de 90° (S3)** es la peor candidata: las cajas de este dataset son verticales (relación w/h entre 0,4 y 1,0 en casi todos los casos), y al rotar aparecen cajas apaisadas que no existen en val ni en test, con lo que se gasta capacidad en una forma que nunca se va a evaluar.

El orden que esperamos, de mayor a menor mejora de mAP@[.5:.95] en test, está en la tabla siguiente, donde cada experimento lleva el rango de mejora previsto.

In [ ]:
BASE_SSD, BASE_CAS = "SSD300 baseline", "Cascade R-CNN baseline"

# Una fila por experimento. Cada uno cambia UNA sola cosa respecto a su referencia ("ref").
# pred_rank / pred_lo / pred_hi: PREDICCIÓN ANTES DE EJECUTAR (posición esperada por mejora de mAP@[.5:.95] en test, y rango
# esperado de la mejora en puntos de mAP). Se compara con el resultado observado en la sección final.
REGISTRY = [
 dict(id="E1", tag="ens_E1_wbf",      exp="E1 Ensemble SSD+Cascade (WBF)", det="Ensemble",      ref="BEST",   change="fusionar con WBF las predicciones de ambos baselines",     pred_rank=1,  pred_lo=0.010, pred_hi=0.030),
 dict(id="C1", tag="cascade_C1_lr",   exp="C1 Cascade LR x10",             det="Cascade R-CNN", ref=BASE_CAS, change="BASE_LR 0,00025 -> 0,0025 (escalado lineal de Detectron2)", pred_rank=2,  pred_lo=0.000, pred_hi=0.030),
 dict(id="S1", tag="ssd_S1_hflip",    exp="S1 SSD volteo horizontal",      det="SSD300",        ref=BASE_SSD, change="volteo horizontal (p = 0,5)",                    pred_rank=3,  pred_lo=0.003, pred_hi=0.020),
 dict(id="S6", tag="ssd_S6_tta_h",    exp="S6 SSD TTA (original + h)",     det="SSD300",        ref=BASE_SSD, change="TTA: original + volteo horizontal, fusión WBF",   pred_rank=4,  pred_lo=0.002, pred_hi=0.015),
 dict(id="S4", tag="ssd_S4_color",    exp="S4 SSD jitter de color",        det="SSD300",        ref=BASE_SSD, change="brillo/contraste/saturación ±25 % (p = 0,5)",     pred_rank=5,  pred_lo=0.000, pred_hi=0.012),
 dict(id="S5", tag="ssd_S5_lr_bb",    exp="S5 SSD LR backbone x0.1",       det="SSD300",        ref=BASE_SSD, change="LR del backbone = 0,1 × LR de la cabeza",         pred_rank=6,  pred_lo=-0.008, pred_hi=0.010),
 dict(id="C3", tag="cascade_C3_color",exp="C3 Cascade jitter de color",    det="Cascade R-CNN", ref=BASE_CAS, change="añadir brillo/contraste/saturación ±25 %",        pred_rank=7,  pred_lo=-0.003, pred_hi=0.007),
 dict(id="S7", tag="ssd_S7_tta_hv",   exp="S7 SSD TTA (original + h + v)", det="SSD300",        ref=BASE_SSD, change="TTA: original + volteo horizontal + vertical, WBF", pred_rank=8,  pred_lo=-0.008, pred_hi=0.008),
 dict(id="C2", tag="cascade_C2_vflip",exp="C2 Cascade volteo vertical",    det="Cascade R-CNN", ref=BASE_CAS, change="añadir volteo vertical (p = 0,5)",                pred_rank=9,  pred_lo=-0.010, pred_hi=0.004),
 dict(id="S2", tag="ssd_S2_vflip",    exp="S2 SSD volteo vertical",        det="SSD300",        ref=BASE_SSD, change="volteo vertical (p = 0,5)",                      pred_rank=10, pred_lo=-0.015, pred_hi=0.004),
 dict(id="S3", tag="ssd_S3_rot90",    exp="S3 SSD rotación 90°",           det="SSD300",        ref=BASE_SSD, change="rotación de 90°/180°/270° (p = 0,5)",             pred_rank=11, pred_lo=-0.040, pred_hi=0.000),
]
CONTROL = dict(tag="ssd_S0_seed", exp="S0 SSD baseline (otra semilla)", seed=SEED + 1)   # control del ruido entre ejecuciones
REG = {r["id"]: r for r in REGISTRY}
display(pd.DataFrame(REGISTRY)[["id", "exp", "ref", "change", "pred_rank", "pred_lo", "pred_hi"]].assign(
    pred_lo=lambda d: (100 * d.pred_lo).round(1), pred_hi=lambda d: (100 * d.pred_hi).round(1)).rename(
    columns={"pred_rank": "rank_previsto", "pred_lo": "Δ mAP previsto mín (pts)", "pred_hi": "Δ mAP previsto máx (pts)"}).set_index("id"))

## 5. Baselines

Se reutilizan los checkpoints de los baselines si existen y se reevalúan siempre, para que los tiempos de inferencia se midan en esta misma sesión y sean comparables con los de los experimentos. El informe tiene el mismo formato que el del cuaderno de baselines.

In [ ]:
# Baselines: se reutilizan los checkpoints del cuaderno de baselines si existen (ver import_previous_work); si no, se entrenan con
# exactamente la misma receta (SSD ≈ 20 min, Cascade R-CNN ≈ 63 min). Se reevalúan siempre para medir los tiempos en ESTA sesión.
run_ssd_experiment("ssd300_baseline", BASE_SSD, reevaluate=True)
run_cascade_experiment("cascade_baseline", BASE_CAS, reevaluate=True)

## 6. Mejoras que solo necesitan inferencia: TTA y ensemble

Estas tres pruebas no entrenan nada: parten de los modelos baseline y se ejecutan primero porque tardan pocos minutos. En las dos versiones de TTA se predice la imagen original y sus volteos, se deshace el volteo en las cajas y se fusionan las predicciones con WBF. En el ensemble se fusionan con WBF las predicciones de SSD y de Cascade R-CNN; los pesos de cada detector y el umbral de IoU de fusión se eligen con una rejilla en validación y el test solo se evalúa con la configuración elegida. El tiempo de inferencia de TTA y ensemble se mide de verdad sobre el proceso completo (varias pasadas más la fusión en CPU) y no se estima multiplicando, así que refleja también el coste de WBF.

In [ ]:
# ---- S6 y S7: TTA sobre el SSD baseline (solo inferencia) ----
def run_tta(rid, flips):
    r = REG[rid]; tag, exp = r["tag"], r["exp"]
    if is_done(tag, exp): print(f"[{tag}] ya hecho: se omite"); return
    model = build_ssd(PRETRAINED).to(DEVICE)
    model.load_state_dict(torch.load(ssd_paths("ssd300_baseline")[0], map_location=DEVICE)); model.eval()
    views = [None] + flips
    def tta_infer(im):                                           # tiempo real de la imagen: n pasadas + WBF
        x = TF.to_tensor(im).to(DEVICE); H, W = x.shape[-2:]; items = []
        for f in views:
            xi = x if f is None else x.flip(-1 if f == "h" else -2)
            items.append(ssd_items(model([xi])[0], W, H, f))
        return wbf_image(items, W, H, iou_thr=0.55)
    ms = time_inference(tta_infer, COCO_GT["test"], CLEAN["test"]["images"])
    for split in ("val", "test"):
        lists = [predict_ssd(model, DS[split], flip=f) for f in views]
        evaluate_and_report(tag, exp, "SSD300", split, fuse_predictions(COCO_GT[split], lists, iou_thr=0.55),
                            ms, ssd_train_time("ssd300_baseline"), count_params(model))
    del model; gc.collect(); torch.cuda.empty_cache()

# ---- E1: ensemble SSD + Cascade (baselines) con WBF; pesos y umbral de fusión elegidos en VAL ----
def run_ensemble():
    r = REG["E1"]; tag, exp = r["tag"], r["exp"]
    if is_done(tag, exp): print(f"[{tag}] ya hecho: se omite"); return
    grid = [(w, thr) for w in ([1, 1], [1, 2], [2, 1]) for thr in (0.5, 0.6, 0.7)]
    pv_ssd, pv_cas = load_preds("ssd300_baseline", "val"), load_preds("cascade_baseline", "val")
    scores = [(evaluate_coco(COCO_GT["val"], fuse_predictions(COCO_GT["val"], [pv_ssd, pv_cas], weights=w, iou_thr=thr))["mAP"], w, thr)
              for w, thr in grid]
    pd.DataFrame(scores, columns=["val_mAP", "pesos[SSD,Cascade]", "iou_thr"]).sort_values("val_mAP", ascending=False).to_csv(
        WORK_DIR / "ensemble_grid_val.csv", index=False)
    _, w, thr = max(scores, key=lambda s: s[0])
    print(f"Mejor combinación en val: pesos [SSD, Cascade] = {w}, iou_thr = {thr}")
    ssd = build_ssd(PRETRAINED).to(DEVICE); ssd.load_state_dict(torch.load(ssd_paths("ssd300_baseline")[0], map_location=DEVICE)); ssd.eval()
    cas = get_cascade_predictor("cascade_baseline")
    def ens_infer(im):                                           # tiempo real: SSD + Cascade + WBF
        x = TF.to_tensor(im).to(DEVICE); H, W = x.shape[-2:]
        return wbf_image([ssd_items(ssd([x])[0], W, H), d2_items(cas(np.ascontiguousarray(im[:, :, ::-1]))["instances"])], W, H, weights=w, iou_thr=thr)
    ms = time_inference(ens_infer, COCO_GT["test"], CLEAN["test"]["images"])
    for split in ("val", "test"):
        fused = fuse_predictions(COCO_GT[split], [load_preds("ssd300_baseline", split), load_preds("cascade_baseline", split)], weights=w, iou_thr=thr)
        evaluate_and_report(tag, exp, "Ensemble", split, fused, ms, ssd_train_time("ssd300_baseline") + cascade_train_time("cascade_baseline"),
                            count_params(ssd) + count_params(cas.model))
    del ssd, cas; gc.collect(); torch.cuda.empty_cache()

### S6 y S7: TTA sobre el SSD baseline

S6 usa la imagen original y su volteo horizontal, que es una transformación que el modelo ha visto en otros detectores y que mantiene la forma de las cajas. S7 añade el volteo vertical, que es la versión del otro cuaderno. Probar las dos por separado permite saber si la vista boca abajo aporta o estorba.

In [ ]:
if RUN["inferencia"]:
    run_tta("S6", ["h"])
    run_tta("S7", ["h", "v"])

### E1: ensemble SSD + Cascade R-CNN con WBF

Un detector de una etapa con anclas y otro de dos etapas en cascada tienen sesgos distintos, de modo que sus errores se solapan solo en parte. La matriz de confusión de los baselines lo respalda: sus aciertos son parecidos (3432 y 3410), pero SSD confunde menos `young` con `Ready` y genera menos falsos positivos, mientras que Cascade deja menos objetos sin detectar. La fusión conserva lo que ambos ven y promedia las cajas, que es lo que más ayuda al mAP a IoU alto.

In [ ]:
if RUN["inferencia"]:
    run_ensemble()

## 7. Mejoras de SSD300 que requieren entrenar

### S0: control de ruido

No es una mejora, sino el baseline de SSD entrenado con otra semilla (cambian la inicialización de la cabeza y el orden de las imágenes). La diferencia con el baseline original indica cuánto varía el mAP solo por azar, y se usa como umbral de ruido en el ranking.

In [ ]:
if RUN["control_seed"]:
    run_ssd_experiment(CONTROL["tag"], CONTROL["exp"], seed=CONTROL["seed"])

### S1: volteo horizontal

La pérdida de validación del baseline deja de bajar en la época 8 mientras la de entrenamiento sigue cayendo, señal de que un dataset de unas mil imágenes se queda corto para ajustar toda la red. Un volteo horizontal duplica los datos efectivos sin alterar la forma de las cajas ni la perspectiva vertical de la escena, y la composición de las bandejas es simétrica de izquierda a derecha.

In [ ]:
if RUN["ssd_aug"]:
    run_ssd_experiment(REG["S1"]["tag"], REG["S1"]["exp"], augment=dict(hflip=0.5))

### S2: volteo vertical

La escena está captada con un gran angular de ojo de pez y con la lámpara en la parte superior de la imagen, de modo que un volteo vertical produce escenas físicamente imposibles y cambia la relación entre la altura en la foto y el tamaño de los objetos. Esperamos que no ayude, pero lo comprobamos porque el otro cuaderno lo daba por bueno, y es la comparación que permite saber si esa hipótesis era válida.

In [ ]:
if RUN["ssd_aug"]:
    run_ssd_experiment(REG["S2"]["tag"], REG["S2"]["exp"], augment=dict(vflip=0.5))

### S3: rotación de 90°

Las cajas del dataset son verticales: la relación de aspecto w/h se concentra entre 0,4 y 1,0. Una rotación de 90° o 270° las convierte en cajas apaisadas, que no aparecen en ninguna imagen de validación ni de test, y además cambia la perspectiva de la escena. Esperamos que sea la transformación más perjudicial y la usamos como contraste: si aun así no empeora, el modelo es más invariante de lo que suponemos.

In [ ]:
if RUN["ssd_aug"]:
    run_ssd_experiment(REG["S3"]["tag"], REG["S3"]["exp"], augment=dict(rot90=0.5))

### S4: variación de color

El tono de la iluminación (muy violácea en unas imágenes y casi natural en otras) cambia mucho entre fotos, y la clase `Ready` se distingue de `young` en parte por el color y el tamaño del follaje. Variar brillo, contraste y saturación un 25 % obliga al modelo a no depender de un tono concreto. Es una transformación que no toca la geometría, por lo que es válida sin discusión, aunque val y test comparten la distribución de iluminación del entrenamiento y la ganancia esperada es pequeña.

In [ ]:
if RUN["ssd_aug"]:
    run_ssd_experiment(REG["S4"]["tag"], REG["S4"]["exp"], augment=dict(color=0.5))

### S5: learning rate diferenciado (backbone × 0,1)

La cabeza de clasificación es nueva y necesita aprender deprisa, mientras que el backbone ya extrae características útiles de COCO y un learning rate alto puede degradarlas. Con un factor de 0,1 el backbone se entrena a 0,0002 y la cabeza a 0,002. Es un término medio entre entrenar todo igual (el baseline) y congelar el backbone, que sería demasiado rígido porque este dominio se parece poco a COCO. El efecto es difícil de anticipar: puede actuar como regularizador y reducir el sobreajuste, pero también frenar una adaptación que sí hace falta.

In [ ]:
if RUN["ssd_lr"]:
    run_ssd_experiment(REG["S5"]["tag"], REG["S5"]["exp"], backbone_lr_mult=0.1)

## 8. Mejoras de Cascade R-CNN

Cada experimento dura unos 63 minutos. Se pueden repartir en varias sesiones quitando identificadores de `CASCADE_EXPERIMENTS`.

### C1: learning rate × 10

El baseline usa un learning rate de 0,00025 con un lote de 2 imágenes, diez veces menos de lo que prescribe el escalado lineal de Detectron2 (0,02 para un lote de 16, es decir, 0,0025 para un lote de 2). A eso se suma que el calentamiento lineal consume las primeras 1000 de las 5300 iteraciones. La curva del baseline respalda la hipótesis de que está infraentrenado: la pérdida seguía bajando y el AP50 en validación subía en la última evaluación. El único cambio es el learning rate; el lote, el número de iteraciones y el calendario son los mismos.

In [ ]:
if RUN["cascade"] and "C1" in CASCADE_EXPERIMENTS:
    run_cascade_experiment(REG["C1"]["tag"], REG["C1"]["exp"], base_lr=0.0025)

### C2: volteo vertical

Se añade a las transformaciones que Detectron2 ya aplicaba (multiescala y volteo horizontal) un volteo vertical con probabilidad 0,5. Es el equivalente en Cascade de S2 y permite comparar si el efecto de esta transformación depende de la arquitectura. Por los mismos motivos geométricos que en SSD esperamos que no ayude.

In [ ]:
if RUN["cascade"] and "C2" in CASCADE_EXPERIMENTS:
    run_cascade_experiment(REG["C2"]["tag"], REG["C2"]["exp"], vflip=True)

### C3: variación de color

Se añade a las transformaciones por defecto la misma variación fotométrica que en S4 (brillo, contraste y saturación de ±25 %). Cascade R-CNN ya aprende con multiescala y volteo, por lo que esperamos un efecto menor que en SSD, aunque permite comprobar si la robustez a la iluminación beneficia por igual a las dos arquitecturas.

In [ ]:
if RUN["cascade"] and "C3" in CASCADE_EXPERIMENTS:
    run_cascade_experiment(REG["C3"]["tag"], REG["C3"]["exp"], color=True)

## 9. Resultados: ranking observado frente al previsto

La tabla ordena los experimentos por su mejora de mAP@[.5:.95] en validación, que es con lo que se decide, e incluye la de test, que es la cifra a reportar. Cada experimento se compara con su referencia: el baseline de su detector y, en el caso del ensemble, el mejor de los dos baselines en validación. El veredicto solo llama «mejora» o «empeora» a lo que supera el ruido medido con el control de semilla tanto en validación como en test; lo demás es no concluyente. Las columnas `Δ_test_por_+100ms` (guardada en el CSV) y `Δms` permiten valorar el coste: las mejoras de entrenamiento no añaden tiempo de inferencia, mientras que el TTA y el ensemble sí.

In [ ]:
res = pd.read_csv(RESULTS_CSV)

def g(exp, split, col):
    r = res[(res.exp == exp) & (res.split == split)]
    return float(r.iloc[0][col]) if len(r) else np.nan

def ref_of(r):
    """Referencia de cada experimento: el baseline de su detector; para el ensemble, el mejor baseline (en val)."""
    return max((BASE_SSD, BASE_CAS), key=lambda b: g(b, "val", "mAP")) if r["ref"] == "BEST" else r["ref"]

# Ruido entre ejecuciones: diferencia entre el baseline SSD y el mismo baseline con otra semilla (si se ejecutó el control)
ctrl = [abs(g(CONTROL["exp"], s, "mAP") - g(BASE_SSD, s, "mAP")) for s in ("val", "test")]
NOISE = max(ctrl) if not np.isnan(ctrl).any() else 0.005
print(f"Umbral de ruido = {100 * NOISE:.2f} pts de mAP  ({'medido con el control de semilla' if not np.isnan(ctrl).any() else 'valor supuesto: no se ejecutó el control'})")

rows = []
for r in REGISTRY:
    if np.isnan(g(r["exp"], "test", "mAP")): continue
    ref = ref_of(r)
    d_val, d_test = (g(r["exp"], s, "mAP") - g(ref, s, "mAP") for s in ("val", "test"))
    d_ms = g(r["exp"], "test", "infer_ms_img") - g(ref, "test", "infer_ms_img")
    verdict = "mejora" if min(d_val, d_test) > NOISE else "empeora" if max(d_val, d_test) < -NOISE else "no concluyente"
    rows.append({"id": r["id"], "experimento": r["exp"], "cambio": r["change"], "referencia": ref,
                 "mAP_val": g(r["exp"], "val", "mAP"), "mAP_test": g(r["exp"], "test", "mAP"), "Δ_val": d_val, "Δ_test": d_test,
                 "ΔmAP50_test": g(r["exp"], "test", "mAP50") - g(ref, "test", "mAP50"),
                 "ΔmAP75_test": g(r["exp"], "test", "mAP75") - g(ref, "test", "mAP75"),
                 "ms_img": g(r["exp"], "test", "infer_ms_img"), "Δms": d_ms, "train_min": g(r["exp"], "test", "train_min"),
                 "Δ_test_por_+100ms": d_test / d_ms * 100 if d_ms > 5 else np.nan,   # NaN = no añade coste de inferencia
                 "rank_previsto": r["pred_rank"], "Δ_previsto": f"{100*r['pred_lo']:+.1f} a {100*r['pred_hi']:+.1f} pts",
                 "dentro_del_rango": r["pred_lo"] <= d_test <= r["pred_hi"], "veredicto": verdict})
rk = pd.DataFrame(rows)
rk["rank_val"] = rk["Δ_val"].rank(ascending=False).astype(int)      # la decisión se toma con val
rk["rank_test"] = rk["Δ_test"].rank(ascending=False).astype(int)    # la cifra a reportar es la de test
rk = rk.sort_values("rank_val").reset_index(drop=True)
rk.to_csv(WORK_DIR / "ranking_mejoras.csv", index=False)

pts = ["mAP_val", "mAP_test", "Δ_val", "Δ_test", "ΔmAP50_test", "ΔmAP75_test"]
show = rk.copy(); show[pts] = (100 * show[pts]).round(2)
display(show[["rank_val", "rank_test", "rank_previsto", "id", "experimento", "mAP_val", "mAP_test", "Δ_val", "Δ_test", "ΔmAP50_test",
              "ΔmAP75_test", "ms_img", "Δms", "veredicto", "Δ_previsto", "dentro_del_rango"]].set_index("rank_val"))
print("(mAP y Δ en puntos porcentuales; Δ respecto a la referencia de cada experimento)")
if len(rk) >= 3:
    print(f"Correlación de Spearman entre el ranking previsto y el observado: val = {rk['rank_previsto'].corr(rk['rank_val'], method='spearman'):.2f}, "
          f"test = {rk['rank_previsto'].corr(rk['rank_test'], method='spearman'):.2f}")

Las tres gráficas siguientes resumen el ranking. A la izquierda, la mejora de cada experimento en validación y en test frente a la banda de ruido; en el centro, la precisión frente al tiempo de inferencia con su frontera de Pareto, donde un punto es mejor cuanto más arriba y más a la izquierda se sitúa; y a la derecha, el cambio del AP por clase, que muestra qué clases se benefician de cada mejora.

In [ ]:
EXP_ORDER = rk.sort_values("Δ_test")
fam_color = {"SSD300": "#1f77b4", "Cascade R-CNN": "#d62728", "Ensemble": "#2ca02c"}
dets = {r["exp"]: r["det"] for r in REGISTRY}
fig, axs = plt.subplots(1, 3, figsize=(23, 6))

# (1) Mejora de mAP@[.5:.95] respecto a la referencia, en val (decisión) y test (cifra a reportar)
y = np.arange(len(EXP_ORDER))
axs[0].barh(y + 0.2, 100 * EXP_ORDER["Δ_test"], 0.4, color=[fam_color[dets[e]] for e in EXP_ORDER["experimento"]], label="test")
axs[0].barh(y - 0.2, 100 * EXP_ORDER["Δ_val"], 0.4, color=[fam_color[dets[e]] for e in EXP_ORDER["experimento"]], alpha=0.45, label="val")
axs[0].axvspan(-100 * NOISE, 100 * NOISE, color="gray", alpha=0.2, label="ruido")
axs[0].axvline(0, color="k", lw=0.8); axs[0].set_yticks(y); axs[0].set_yticklabels(EXP_ORDER["experimento"], fontsize=8)
axs[0].set_xlabel("Δ mAP@[.5:.95] (puntos) respecto a la referencia"); axs[0].set_title("Mejora por experimento (oscuro = test, claro = val)")
axs[0].grid(axis="x", alpha=0.3); axs[0].legend(fontsize=8, loc="lower right")

# (2) Precisión frente a coste de inferencia (test): arriba a la izquierda es mejor
pts_df = res[(res.split == "test") & (res.exp.isin([BASE_SSD, BASE_CAS] + [r["exp"] for r in REGISTRY]))].drop_duplicates("exp")
for _, r in pts_df.iterrows():
    c = fam_color.get(dets.get(r.exp, r.detector), "gray")
    axs[1].scatter(r.infer_ms_img, 100 * r.mAP, s=70, color=c, marker="s" if "baseline" in r.exp else "o", zorder=3)
    axs[1].annotate(r.exp.split(" ")[0] if r.exp[0] in "SCE" and r.exp[1].isdigit() else r.exp.replace(" baseline", " base"),
                    (r.infer_ms_img, 100 * r.mAP), textcoords="offset points", xytext=(4, 5), fontsize=8)
front = pts_df.sort_values("infer_ms_img"); best = -1; fx, fy = [], []
for _, r in front.iterrows():
    if r.mAP > best: best = r.mAP; fx.append(r.infer_ms_img); fy.append(100 * r.mAP)
axs[1].step(fx, fy, where="post", color="k", lw=0.8, ls="--", label="frontera de Pareto"); axs[1].legend(fontsize=8)
axs[1].set_xlabel("tiempo de inferencia (ms/img, batch 1)"); axs[1].set_ylabel("mAP@[.5:.95] test (%)"); axs[1].set_title("Precisión frente a velocidad"); axs[1].grid(alpha=0.3)

# (3) Cambio del AP por clase (test) respecto a la referencia: dónde ayuda cada mejora
cls_rows = []
for _, r in rk.iterrows():
    cls_rows.append([100 * (g(r["experimento"], "test", f"AP_{c}") - g(r["referencia"], "test", f"AP_{c}")) for c in CLASS_NAMES])
hm = np.array(cls_rows); lim = max(1, np.nanmax(np.abs(hm)))
im = axs[2].imshow(hm, cmap="RdYlGn", vmin=-lim, vmax=lim, aspect="auto")
axs[2].set_xticks(range(K)); axs[2].set_xticklabels(CLASS_NAMES, rotation=30); axs[2].set_yticks(range(len(rk))); axs[2].set_yticklabels(rk["experimento"], fontsize=8)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]): axs[2].text(j, i, f"{hm[i, j]:+.1f}", ha="center", va="center", fontsize=8)
axs[2].set_title("Δ AP por clase en test (puntos)"); plt.colorbar(im, ax=axs[2], fraction=0.046)
plt.tight_layout(); plt.savefig(WORK_DIR / "figs" / "mejoras_resumen.png", dpi=130); plt.show()
pd.DataFrame(hm, index=rk["experimento"], columns=CLASS_NAMES).round(2).to_csv(WORK_DIR / "delta_ap_por_clase_test.csv")

### Curvas de aprendizaje

Las curvas de SSD muestran la pérdida de validación, la de entrenamiento (que con aumento de datos no es comparable con la del baseline, porque las imágenes son más difíciles) y el mAP en validación por época. Las de Cascade R-CNN muestran la pérdida de entrenamiento y el AP50 en validación que Detectron2 registra durante el entrenamiento.

In [ ]:
# Curvas de aprendizaje de SSD: pérdida en val y mAP en val por época (solo hay mAP por época en los experimentos de esta sesión)
fig, axs = plt.subplots(1, 3, figsize=(21, 4.6))
ssd_tags = [("ssd300_baseline", "S0 baseline")] + [(CONTROL["tag"], "S0 otra semilla")] + [(r["tag"], r["id"]) for r in REGISTRY if r["tag"].startswith("ssd_S") and r["id"] in ("S1", "S2", "S3", "S4", "S5")]
for tag, lab in ssd_tags:
    p = ssd_paths(tag)[1]
    if not p.exists(): continue
    h = pd.DataFrame(json.loads(p.read_text())["history"])
    axs[0].plot(h.epoch, h.val_loss, label=lab); axs[1].plot(h.epoch, h.train_loss, label=lab)
    if "val_mAP" in h: axs[2].plot(h.epoch, 100 * h.val_mAP, label=lab)
for ax, t in zip(axs, ("SSD: pérdida de validación", "SSD: pérdida de entrenamiento (con aumento no es comparable)", "SSD: mAP@[.5:.95] en val por época (%)")):
    ax.set_title(t, fontsize=10); ax.set_xlabel("época"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(WORK_DIR / "figs" / "mejoras_curvas_ssd.png", dpi=130); plt.show()

# Curvas de Cascade R-CNN: pérdida de entrenamiento (media móvil) y AP50 en val (lo escribe Detectron2 en metrics.json)
fig, axs = plt.subplots(1, 2, figsize=(14, 4.6))
for tag, lab in [("cascade_baseline", "C0 baseline")] + [(r["tag"], r["id"]) for r in REGISTRY if r["tag"].startswith("cascade_C")]:
    mj = WORK_DIR / "d2" / tag / "metrics.json"
    if not mj.exists(): continue
    rows_ = [json.loads(l) for l in mj.read_text().splitlines() if l.strip()]
    ipe = len(DS["train"]) / 2
    tr = pd.DataFrame([{"epoch": r["iteration"] / ipe, "loss": r["total_loss"]} for r in rows_ if "total_loss" in r])
    axs[0].plot(tr.epoch, tr.loss.rolling(10, min_periods=1).mean(), label=lab)
    val = [(r["iteration"] / ipe, r["bbox/AP50"]) for r in rows_ if "bbox/AP50" in r]
    if val: axs[1].plot(*zip(*val), "o-", label=lab)
axs[0].set_title("Cascade R-CNN: pérdida de entrenamiento"); axs[1].set_title("Cascade R-CNN: AP50 en val durante el entrenamiento")
for ax in axs: ax.set_xlabel("época"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(WORK_DIR / "figs" / "mejoras_curvas_cascade.png", dpi=130); plt.show()

### Análisis de fallos

Las curvas precisión-recall comparan los baselines con las tres mejores configuraciones, a IoU 0,5 y 0,75. Las matrices de confusión y los ejemplos usan un umbral de confianza de 0,5 y emparejan por IoU ≥ 0,5; permiten ver si una mejora reduce las confusiones entre clases, los objetos no detectados o los falsos positivos. Los ejemplos son las tres imágenes de test con más errores en los baselines.

In [ ]:
SCORE_VIS = 0.5     # umbral de confianza del punto de operación para el análisis de fallos
IOU_MATCH = 0.5

def group_by_image(preds, score_thr=SCORE_VIS):
    """Predicciones con score >= umbral, agrupadas por imagen y ordenadas de mayor a menor confianza."""
    g = defaultdict(list)
    for p in preds:
        if p["score"] >= score_thr: g[p["image_id"]].append(p)
    for v in g.values(): v.sort(key=lambda p: -p["score"])
    return g

def box_iou_xywh(a, b):
    """IoU entre dos conjuntos de cajas [x, y, w, h] -> matriz (len(a), len(b))."""
    ax2, ay2, bx2, by2 = a[:, 0] + a[:, 2], a[:, 1] + a[:, 3], b[:, 0] + b[:, 2], b[:, 1] + b[:, 3]
    iw = np.clip(np.minimum(ax2[:, None], bx2[None]) - np.maximum(a[:, 0][:, None], b[:, 0][None]), 0, None)
    ih = np.clip(np.minimum(ay2[:, None], by2[None]) - np.maximum(a[:, 1][:, None], b[:, 1][None]), 0, None)
    inter = iw * ih
    union = (a[:, 2] * a[:, 3])[:, None] + (b[:, 2] * b[:, 3])[None] - inter
    return inter / np.maximum(union, 1e-9)

def detection_confusion(coco_gt, by_img, image_ids=None, iou_thr=IOU_MATCH):
    """Matriz de confusión de detección: filas = clase real, columnas = clase predicha, y una fila/columna extra de 'fondo'.
    Cada detección (de mayor a menor confianza) se empareja con el objeto real libre de mayor IoU (>= iou_thr), sin mirar la clase.
    Un objeto real sin detección cuenta en la columna 'fondo' (no detectado) y una detección sin objeto real, en la fila 'fondo'."""
    n = len(CLASS_NAMES); cm = np.zeros((n + 1, n + 1), dtype=int)
    for iid in (image_ids if image_ids is not None else coco_gt.imgs):
        gts = coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=iid)); dts = by_img.get(iid, [])
        gt_cls = [CID2IDX[g["category_id"]] for g in gts]
        iou = box_iou_xywh(np.array([d["bbox"] for d in dts]).reshape(-1, 4),
                           np.array([g["bbox"] for g in gts]).reshape(-1, 4)) if dts and gts else None
        used = set()
        for i, d in enumerate(dts):
            j = -1
            if iou is not None:
                free = [g for g in range(len(gts)) if g not in used and iou[i, g] >= iou_thr]
                if free: j = max(free, key=lambda g: iou[i, g])
            if j >= 0:
                used.add(j); cm[gt_cls[j], CID2IDX[d["category_id"]]] += 1
            else:
                cm[n, CID2IDX[d["category_id"]]] += 1
        for g in range(len(gts)):
            if g not in used: cm[gt_cls[g], n] += 1
    return cm

In [ ]:
# ---- Curvas precisión-recall (media de las 5 clases): baselines y los mejores experimentos ----
best_ids = rk.sort_values("Δ_test", ascending=False).head(3)["experimento"].tolist()
CURVES = [(BASE_SSD, "ssd300_baseline")] + [(BASE_CAS, "cascade_baseline")] + [(r["exp"], r["tag"]) for r in REGISTRY if r["exp"] in best_ids]
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for exp, tag in CURVES:
    if not (WORK_DIR / "preds" / f"{tag}_test.json").exists(): continue
    ev = run_cocoeval(COCO_GT["test"], load_preds(tag, "test"))
    for ax, thr in zip(axs, (0.50, 0.75)):
        t = int(np.argmin(np.abs(ev.params.iouThrs - thr)))
        p = ev.eval["precision"][t, :, :, 0, 2].astype(float); p[p < 0] = np.nan
        ax.plot(ev.params.recThrs, np.nanmean(p, axis=1), lw=2.5 if "baseline" in exp else 1.6, ls="--" if "baseline" in exp else "-",
                label=f"{exp} (AP={100 * _mean_valid(p):.1f})")
for ax, thr in zip(axs, (0.50, 0.75)):
    ax.set_title(f"Curva PR media de las clases, IoU = {thr:.2f} (test)"); ax.set_xlabel("recall"); ax.set_ylabel("precisión"); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.savefig(WORK_DIR / "figs" / "mejoras_pr.png", dpi=130); plt.show()

# ---- Matrices de confusión (score >= 0.5, IoU >= 0.5): baselines, mejor experimento individual y ensemble ----
indiv = rk[rk["experimento"].isin([r["exp"] for r in REGISTRY if r["id"] != "E1"])].sort_values("Δ_test", ascending=False)
MATS = {BASE_SSD: "ssd300_baseline", BASE_CAS: "cascade_baseline"}
if len(indiv): MATS[indiv.iloc[0]["experimento"] + "  (mejor individual)"] = REG[indiv.iloc[0]["id"]]["tag"]
if (WORK_DIR / "preds" / "ens_E1_wbf_test.json").exists(): MATS[REG["E1"]["exp"]] = "ens_E1_wbf"
cms, summ = {}, []
for name, tag in MATS.items():
    cms[name] = cm = detection_confusion(COCO_GT["test"], group_by_image(load_preds(tag, "test"))); tp = int(np.trace(cm[:K, :K]))
    summ.append({"experimento": name, "aciertos": tp, "recall": tp / cm[:K].sum(), "precisión": tp / cm[:, :K].sum(),
                 "clase confundida": int(cm[:K, :K].sum() - tp), "no detectados": int(cm[:K, K].sum()), "falsos positivos": int(cm[K, :K].sum())})
display(pd.DataFrame(summ).set_index("experimento").round(3))
fig, axs = plt.subplots(1, len(cms), figsize=(6.2 * len(cms), 5.8), squeeze=False)
for ax, (name, cm) in zip(axs[0], cms.items()):
    norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1); ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(K + 1):
        for j in range(K + 1): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if norm[i, j] > 0.5 else "black", fontsize=8)
    ax.set_xticks(range(K + 1)); ax.set_xticklabels(CLASS_NAMES + ["fondo"], rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(K + 1)); ax.set_yticklabels(CLASS_NAMES + ["fondo"], fontsize=8); ax.set_title(name, fontsize=9)
plt.tight_layout(); plt.savefig(WORK_DIR / "figs" / "mejoras_confusion.png", dpi=130); plt.show()

# ---- Ejemplos cualitativos: las 3 imágenes de test con más errores de los baselines, frente a las mejores configuraciones ----
def draw_boxes(ax, boxes, cids, scores=None):
    for i, (x, y, w, h) in enumerate(boxes):
        c = CLASS_COLORS[CID2IDX[cids[i]]]; ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, lw=1.6, ec=c))
        if scores is not None: ax.text(x, max(y - 3, 8), f"{CLASS_NAMES[CID2IDX[cids[i]]]} {scores[i]:.2f}", fontsize=6, color="white", bbox=dict(facecolor=c, alpha=0.75, pad=0.4, lw=0))
coco_t = COCO_GT["test"]; grouped = {n: group_by_image(load_preds(t, "test")) for n, t in MATS.items()}
def n_err(name, iid):
    cm = detection_confusion(coco_t, grouped[name], [iid]); return int(cm.sum() - np.trace(cm[:K, :K]))
tot = {iid: n_err(BASE_SSD, iid) + n_err(BASE_CAS, iid) for iid in coco_t.imgs}
cases = sorted(tot, key=tot.get, reverse=True)[:3]
fig, axs = plt.subplots(len(cases), 1 + len(MATS), figsize=(4.6 * (1 + len(MATS)), 4.8 * len(cases)), squeeze=False)
for r, iid in enumerate(cases):
    info = coco_t.imgs[iid]; img = Image.open(CLEAN["test"]["images"] / info["file_name"]).convert("RGB"); anns = coco_t.loadAnns(coco_t.getAnnIds(imgIds=iid))
    axs[r, 0].imshow(img); draw_boxes(axs[r, 0], [a["bbox"] for a in anns], [a["category_id"] for a in anns]); axs[r, 0].set_title(f"imagen {iid} · ground truth ({len(anns)})", fontsize=9)
    for c, name in enumerate(MATS, start=1):
        dts = grouped[name].get(iid, []); axs[r, c].imshow(img)
        draw_boxes(axs[r, c], [d["bbox"] for d in dts], [d["category_id"] for d in dts], [d["score"] for d in dts]); axs[r, c].set_title(f"{name} · {n_err(name, iid)} errores", fontsize=8)
    for ax in axs[r]: ax.axis("off")
fig.legend(handles=[patches.Patch(color=CLASS_COLORS[i], label=n) for i, n in enumerate(CLASS_NAMES)], loc="lower center", ncol=K)
plt.tight_layout(rect=(0, 0.02, 1, 1)); plt.savefig(WORK_DIR / "figs" / "mejoras_cualitativo.png", dpi=110); plt.show()

## 10. Resumen y conclusiones

La celda siguiente compara automáticamente lo observado con lo previsto en la sección 4. Las conclusiones de la memoria (qué mejoró, qué no y por qué) deben escribirse a la vista de estos números y de las figuras anteriores: contrastando la hipótesis de cada experimento con la que se formuló antes de ejecutarlo, y recordando que cada configuración se ha entrenado con una sola semilla.

In [ ]:
print("RESUMEN AUTOMÁTICO (se regenera con los resultados de cada ejecución)\n")
for _, r in rk.sort_values("Δ_test", ascending=False).iterrows():
    print(f"{r['id']:>2} {r['experimento']:<36} Δ test = {100*r['Δ_test']:+5.2f} pts | Δ val = {100*r['Δ_val']:+5.2f} pts | previsto {r['Δ_previsto']:>16} "
          f"({'dentro' if r['dentro_del_rango'] else 'fuera'} del rango) | {r['veredicto']}")
ok_sign = ((rk["Δ_test"] > 0) == rk["id"].map(lambda i: (REG[i]["pred_lo"] + REG[i]["pred_hi"]) > 0)).sum()
print(f"\nSigno de la mejora acertado en {ok_sign} de {len(rk)} experimentos; resultado dentro del rango previsto en {int(rk['dentro_del_rango'].sum())} de {len(rk)}.")
mejoras = rk[rk["veredicto"] == "mejora"]
print("Mejoras claras (superan el ruido en val y test):", ", ".join(mejoras["id"]) if len(mejoras) else "ninguna")
gratis = mejoras[mejoras["Δms"] < 5]
print("De ellas, sin coste de inferencia:", ", ".join(gratis["id"]) if len(gratis) else "ninguna")
print(f"Ruido entre ejecuciones usado como umbral: {100*NOISE:.2f} pts. Una sola semilla por experimento: las diferencias menores que ese umbral no son concluyentes.")

**Siguiente paso, fuera de este estudio:** una vez conocido qué mejoras funcionan por separado, se pueden combinar las que superen el ruido (por ejemplo, volteo horizontal y color en SSD, o el learning rate de Cascade con un aumento válido) y aplicar TTA o ensemble sobre los modelos resultantes. Se deja fuera de este cuaderno porque mezclaría varias mejoras en un mismo experimento.